# §6.3 Case Study 1 — Splitwise vs Co-located

Loads the parametric sweep produced by `tools/experiments/run_splitwise_sweep.py` and produces:

1. **Fig 6** — TTFT P99 across P:D ratios for 3 workload shapes (the main "when does Splitwise win?" figure)
2. **Fig 7** — Throughput–latency Pareto curves per workload
3. **Fig 8** — KV bandwidth sensitivity (the "NDR InfiniBand isn't required" claim)

All figures are saved to `figures/` as PDF + PNG, ready for paper inclusion.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

FIGDIR = Path('figures'); FIGDIR.mkdir(exist_ok=True)
RESULTS = '../experiments/results/splitwise_sweep.csv'

plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 300,
    'font.size': 10, 'axes.titlesize': 11, 'axes.labelsize': 10,
    'legend.fontsize': 9, 'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'axes.spines.top': False, 'axes.spines.right': False,
})

df = pd.read_csv(RESULTS)
df['pd_ratio'] = df.apply(lambda r: f"{int(r.prefill_gpus)}:{int(r.decode_gpus)}", axis=1)
df['pd_share_prefill'] = df.prefill_gpus / (df.prefill_gpus + df.decode_gpus)
print(f'loaded {len(df)} cells from {RESULTS}')
df.head(3)

## Fig 6 — TTFT P99 vs P:D ratio (3 workload shapes)

**Story:** for short prompts, Splitwise gains nothing (decode bottleneck). For long prompts, the right P:D ratio (≥ 6:2) cuts P99 TTFT 6× vs the worst Splitwise ratio.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 4.2))

WORKLOADS = ['short', 'medium', 'long']
MARKERS   = {'short': 'o', 'medium': 's', 'long': '^'}
COLORS    = {'short': '#1f77b4', 'medium': '#2ca02c', 'long': '#d62728'}

# Pick a representative KV bw (200 GB/s = HDR InfiniBand) for the main figure;
# Fig 8 will show that this choice barely matters.
view = df[(df['mode'] == 'splitwise') & (df.kv_bw_gbs == 200)]

for w in WORKLOADS:
    sub = view[view.workload == w].sort_values('pd_share_prefill')
    ax.plot(sub.pd_share_prefill, sub.p99_ttft_s,
            marker=MARKERS[w], color=COLORS[w],
            label=f'splitwise — {w}', linewidth=1.6)
    # Co-located baseline as horizontal dashed line per workload
    base = df[(df['mode'] == 'colocated') & (df.workload == w)]
    if not base.empty:
        ax.axhline(base.p99_ttft_s.mean(), color=COLORS[w], linestyle=':', alpha=0.6,
                   label=f'colocated — {w}')

ax.set_xlabel('prefill GPU share  =  prefill / (prefill + decode)')
ax.set_ylabel('P99 TTFT (s)')
ax.set_title('Fig 6 — TTFT P99 across P:D ratios (8 GPUs total, KV bw 200 GB/s)')
ax.set_yscale('log')
ax.grid(alpha=0.3, which='both')
ax.legend(loc='upper right', ncols=2, frameon=False)
fig.tight_layout()
fig.savefig(FIGDIR / 'fig6_ttft_vs_pd_ratio.pdf')
fig.savefig(FIGDIR / 'fig6_ttft_vs_pd_ratio.png')

## Fig 7 — Throughput–latency Pareto per workload

**Story:** show the achievable Pareto frontier (mean E2E vs SLO attainment) and where each P:D ratio sits. Co-located baseline appears as a single point per workload.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, w in zip(axes, WORKLOADS):
    view = df[(df.workload == w) & (df.kv_bw_gbs == 200)]
    sw = view[view['mode'] == 'splitwise'].sort_values('pd_share_prefill')
    co = view[view['mode'] == 'colocated']

    sc = ax.scatter(sw.mean_e2e_s, 100*sw.slo_attainment, c=sw.prefill_gpus,
                    cmap='viridis', s=85, edgecolor='k', linewidth=0.5)
    if not co.empty:
        ax.scatter(co.mean_e2e_s, 100*co.slo_attainment, c='red',
                   marker='*', s=180, label='colocated', edgecolor='k', zorder=5)
    # Annotate prefill-GPU count next to each splitwise point
    for _, r in sw.iterrows():
        ax.annotate(f"{int(r.prefill_gpus)}p{int(r.decode_gpus)}d",
                    (r.mean_e2e_s, 100*r.slo_attainment),
                    fontsize=7, xytext=(4, 4), textcoords='offset points')
    ax.set_xlabel('Mean E2E latency (s)')
    ax.set_ylabel('SLO attainment (%)')
    ax.set_title(f'{w} workload')
    ax.grid(alpha=0.3)
    if w == WORKLOADS[-1]:
        cb = plt.colorbar(sc, ax=ax, fraction=0.045, label='prefill GPUs')
        ax.legend(loc='lower right', frameon=False)

fig.suptitle('Fig 7 — Throughput–latency Pareto per workload', y=1.02)
fig.tight_layout()
fig.savefig(FIGDIR / 'fig7_pareto.pdf')
fig.savefig(FIGDIR / 'fig7_pareto.png')

## Fig 8 — KV bandwidth sensitivity

**Story:** the "NDR InfiniBand at 400 GB/s gives < 1% additional gain over HDR 200 GB/s" finding. We compare TTFT P99 across 100/200/400 GB/s for the *best* P:D ratio per workload.

In [ ]:
# For each (workload), pick the P:D ratio that minimizes P99 TTFT at kv_bw=200, then plot all 3 KV BW points.
fig, ax = plt.subplots(figsize=(7, 4.2))

BW_VALUES = sorted(df[df['mode'] == 'splitwise'].kv_bw_gbs.unique())
x = np.arange(len(BW_VALUES))
width = 0.25

for i, w in enumerate(WORKLOADS):
    ref = df[(df.workload == w) & (df['mode'] == 'splitwise') & (df.kv_bw_gbs == 200)]
    if ref.empty: continue
    best_pd = ref.loc[ref.p99_ttft_s.idxmin(), 'pd_ratio']
    sub = df[(df.workload == w) & (df['mode'] == 'splitwise') & (df.pd_ratio == best_pd)].sort_values('kv_bw_gbs')
    ax.bar(x + i*width - width, sub.p99_ttft_s.values, width=width,
           color=COLORS[w], label=f'{w} (best={best_pd})', edgecolor='k', linewidth=0.4)

ax.set_xticks(x)
ax.set_xticklabels([f'{int(v)} GB/s' for v in BW_VALUES])
ax.set_ylabel('P99 TTFT (s)')
ax.set_xlabel('Inter-host KV transfer bandwidth')
ax.set_title('Fig 8 — KV bandwidth sensitivity at the best P:D ratio per workload')
ax.grid(axis='y', alpha=0.3)
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIGDIR / 'fig8_kvbw_sensitivity.pdf')
fig.savefig(FIGDIR / 'fig8_kvbw_sensitivity.png')

## Summary table for §6.3

Picks the best Splitwise P:D ratio per workload (at KV bw=200) and contrasts it with the co-located baseline.

In [ ]:
rows = []
for w in WORKLOADS:
    co = df[(df.workload == w) & (df['mode'] == 'colocated')]
    sw = df[(df.workload == w) & (df['mode'] == 'splitwise') & (df.kv_bw_gbs == 200)]
    if co.empty or sw.empty: continue
    best = sw.loc[sw.p99_ttft_s.idxmin()]
    rows.append({
        'workload': w,
        'best_pd': best.pd_ratio,
        'colocated_p99_ttft': co.p99_ttft_s.mean(),
        'splitwise_p99_ttft': best.p99_ttft_s,
        'p99_ttft_speedup': co.p99_ttft_s.mean() / best.p99_ttft_s,
        'colocated_e2e': co.mean_e2e_s.mean(),
        'splitwise_e2e': best.mean_e2e_s,
        'colocated_slo': 100*co.slo_attainment.mean(),
        'splitwise_slo': 100*best.slo_attainment,
        'colocated_energy_kwh': co.total_energy_kwh.mean(),
        'splitwise_energy_kwh': best.total_energy_kwh,
    })
summary = pd.DataFrame(rows).set_index('workload')
summary

In [ ]:
summary.to_csv(FIGDIR / 'table_case_study_1.csv')
with open(FIGDIR / 'table_case_study_1.tex', 'w') as f:
    f.write(summary.to_latex(float_format='%.2f'))
print('wrote', FIGDIR / 'table_case_study_1.tex')

## Sanity checks (run before publishing)
Reviewers will probe these. Fail-fast assertions:

In [ ]:
issues = []

# 1. All cells finished a non-trivial fraction of submitted requests
min_finished = df.n_finished.min() / df.requests.iloc[0]
if min_finished < 0.8:
    issues.append(f'some cells dropped > 20% of requests (min finished fraction = {min_finished:.2f})')

# 2. TTFT must be non-negative
if (df.mean_ttft_s < 0).any() or (df.p99_ttft_s < 0).any():
    issues.append('negative TTFT detected — check arrival gating')

# 3. P99 TTFT >= mean TTFT
if not (df.p99_ttft_s >= df.mean_ttft_s).all():
    issues.append('P99 < mean for some cells — check percentile computation')

# 4. E2E must be >= TTFT
if not (df.mean_e2e_s >= df.mean_ttft_s - 1e-3).all():
    issues.append('mean E2E < mean TTFT — first-token after finish is impossible')

if issues:
    print('\u26a0  data quality concerns:')
    for i in issues: print('  -', i)
else:
    print('\u2705 all sanity checks pass')